TASK 1

In [ ]:
def analyze_transactions(transactions):
    customers = {}
    ids = []

    for t in transactions:
        if t["id"] in ids:
            raise ValueError("Duplicate transaction ID")

        ids.append(t["id"])

        name = t["customer"]

        if name not in customers:
            customers[name] = {
                "total": 0,
                "categories": {}
            }

        if t["status"] == "success":
            customers[name]["total"] += t["amount"]

            category = t["category"]

            if category not in customers[name]["categories"]:
                customers[name]["categories"][category] = 0

            customers[name]["categories"][category] += t["amount"]

    result = []

    for name in customers:
        total = customers[name]["total"]
        categories = customers[name]["categories"]

        top_category = None

        for category in categories:
            if top_category is None or categories[category] > categories[top_category]:
                top_category = category

        result.append({
            "customer": name,
            "total": total,
            "top_category": top_category
        })

    result.sort(key=lambda x: (-x["total"], x["customer"]))

    for i in range(len(result)):
        result[i]["rank"] = i + 1

    return result


transactions = [
    {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"},
    {"id": 2, "customer": "Ali", "category": "Travel", "amount": 200, "status": "failed"},
    {"id": 3, "customer": "Sara", "category": "Food", "amount": 700, "status": "success"}
]

answer = analyze_transactions(transactions)

print("Output totals:", end=" ")

for i in range(len(answer)):
    print(answer[i]["customer"] + "=" + str(answer[i]["total"]), end="")

    if i < len(answer) - 1:
        print(", ", end="")

{'customer': 'Sara', 'total': 700, 'top_category': 'Food', 'rank': 1}
{'customer': 'Ali', 'total': 500, 'top_category': 'Food', 'rank': 2}


TASK 2

In [9]:
def longest_unique_window(text):
    last_seen = {}
    start = 0
    longest_start = 0
    longest_length = 0

    for i in range(len(text)):
        char = text[i]

        if char in last_seen and last_seen[char] >= start:
            start = last_seen[char] + 1

        last_seen[char] = i

        current_length = i - start + 1

        if current_length > longest_length:
            longest_length = current_length
            longest_start = start

    substring = text[longest_start:longest_start + longest_length]

    return substring, longest_start, longest_length


print(longest_unique_window("abcabcbb"))
print(longest_unique_window(""))

('abc', 0, 3)
('', 0, 0)


TASK 3

In [10]:
def find_paths(data, target):
    paths = []

    def search(value, path):
        if isinstance(value, dict):
            for key in value:
                if path == "":
                    new_path = key
                else:
                    new_path = path + "." + key

                search(value[key], new_path)

        elif isinstance(value, list):
            for i in range(len(value)):
                new_path = path + "[" + str(i) + "]"
                search(value[i], new_path)

        else:
            if type(value) == type(target) and value == target:
                paths.append(path)

    search(data, "")

    return paths


data = {
    "a": [
        {"x": 5},
        5
    ],
    "b": {
        "y": 5
    }
}

print(find_paths(data, 5))

['a[0].x', 'a[1]', 'b.y']


TASK 4


In [4]:
def execution_order(tasks, dependencies):
    graph = {}
    count = {}

    for task in tasks:
        graph[task] = []
        count[task] = 0

    for a, b in dependencies:
        if a not in graph or b not in graph:
            raise ValueError("Unknown task name")

        graph[a].append(b)
        count[b] += 1

    available = []

    for task in tasks:
        if count[task] == 0:
            available.append(task)

    available.sort()

    result = []

    while len(available) > 0:
        current = available.pop(0)
        result.append(current)

        for next_task in graph[current]:
            count[next_task] -= 1

            if count[next_task] == 0:
                available.append(next_task)

        available.sort()

    if len(result) != len(tasks):
        raise ValueError("Cycle detected in dependencies")

    return result


tasks = ["build", "test", "deploy"]

dependencies = [
    ("build", "test"),
    ("test", "deploy")
]

print(execution_order(tasks, dependencies))

['build', 'test', 'deploy']


TASK 5

In [5]:
import re


def calculate(expression):
    tokens = re.findall(r'\d+|[+\-*/()]|\S', expression)
    position = 0

    def expression_part():
        nonlocal position

        result = term()

        while position < len(tokens) and tokens[position] in ["+", "-"]:
            operator = tokens[position]
            position += 1

            number = term()

            if operator == "+":
                result = result + number
            else:
                result = result - number

        return result

    def term():
        nonlocal position

        result = factor()

        while position < len(tokens) and tokens[position] in ["*", "/"]:
            operator = tokens[position]
            position += 1

            number = factor()

            if operator == "*":
                result = result * number
            else:
                if number == 0:
                    raise ZeroDivisionError("Cannot divide by zero")

                result = result / number

        return result

    def factor():
        nonlocal position

        if position >= len(tokens):
            raise ValueError("Invalid expression")

        token = tokens[position]

        if token == "-":
            position += 1
            return -factor()

        if token == "+":
            position += 1
            return factor()

        if token == "(":
            position += 1
            result = expression_part()

            if position >= len(tokens) or tokens[position] != ")":
                raise ValueError("Missing closing parenthesis")

            position += 1
            return result

        if token.isdigit():
            position += 1
            return int(token)

        raise ValueError("Invalid expression")

    answer = expression_part()

    if position != len(tokens):
        raise ValueError("Invalid expression")

    return answer


expression = "-(3 + 2) * 4 + 18 / 3"

print(calculate(expression))

-14.0


TASK 6

In [11]:
from collections import namedtuple
from uuid import uuid4


Transaction = namedtuple(
    "Transaction",
    ["transaction_id", "action", "amount", "balance"]
)


class Account:
    def __init__(self, account_id, balance):
        self.account_id = account_id
        self.__balance = balance
        self.transactions = []

    @property
    def balance(self):
        return self.__balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")

        self.__balance += amount

        transaction = Transaction(
            str(uuid4()),
            "Deposit",
            amount,
            self.__balance
        )

        self.transactions.append(transaction)

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")

        if amount > self.__balance:
            raise ValueError("Insufficient funds")

        self.__balance -= amount

        transaction = Transaction(
            str(uuid4()),
            "Withdrawal",
            amount,
            self.__balance
        )

        self.transactions.append(transaction)

    def statement(self):
        for transaction in self.transactions:
            print(transaction)


class Bank:
    def __init__(self):
        self.accounts = {}

    def add_account(self, account):
        if account.account_id in self.accounts:
            raise ValueError("Account already exists")

        self.accounts[account.account_id] = account

    def transfer(self, from_id, to_id, amount):
        if from_id not in self.accounts or to_id not in self.accounts:
            raise ValueError("Account does not exist")

        if amount <= 0:
            raise ValueError("Amount must be positive")

        sender = self.accounts[from_id]
        receiver = self.accounts[to_id]

        if sender.balance < amount:
            raise ValueError("Insufficient funds")

        sender.withdraw(amount)
        receiver.deposit(amount)


bank = Bank()

account_a = Account("A", 1000)
account_b = Account("B", 200)

bank.add_account(account_a)
bank.add_account(account_b)

bank.transfer("A", "B", 350)

print("Account A:", account_a.balance)
print("Account B:", account_b.balance)

Account A: 650
Account B: 550


TASK 7

In [12]:
from abc import ABC, abstractmethod


class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def calculate_pay(self):
        pass


class SalariedEmployee(Employee):
    def __init__(self, name, salary):
        super().__init__(name)
        self.salary = salary

    def calculate_pay(self):
        return self.salary


class HourlyEmployee(Employee):
    def __init__(self, name, rate, hours):
        super().__init__(name)
        self.rate = rate
        self.hours = hours

    def calculate_pay(self):
        if self.hours > 40:
            normal_pay = 40 * self.rate
            overtime_pay = (self.hours - 40) * self.rate * 1.5

            return normal_pay + overtime_pay

        return self.hours * self.rate


class CommissionEmployee(Employee):
    def __init__(self, name, base, sales, rate):
        super().__init__(name)
        self.base = base
        self.sales = sales
        self.rate = rate

    def calculate_pay(self):
        return self.base + (self.sales * self.rate)


def payroll_summary(employees):
    for employee in employees:
        print(employee.name, ":", employee.calculate_pay())


employees = [
    SalariedEmployee("Ali", 50000),
    HourlyEmployee("Ahmed", 20, 45),
    CommissionEmployee("Sara", 500, 4000, 0.10)
]

payroll_summary(employees)

Ali : 50000
Ahmed : 950.0
Sara : 900.0


TASK 8

In [13]:
import time
from functools import wraps


def ttl_cache(seconds):
    def decorator(function):
        cache = {}
        hits = 0
        misses = 0

        @wraps(function)
        def wrapper(*args, **kwargs):
            nonlocal hits, misses

            key = (args, tuple(sorted(kwargs.items())))

            try:
                hash(key)
            except TypeError:
                raise TypeError("Arguments must be hashable")

            current_time = time.monotonic()

            if key in cache:
                result, expiry_time = cache[key]

                if current_time < expiry_time:
                    hits += 1
                    return result

            misses += 1

            result = function(*args, **kwargs)

            expiry_time = current_time + seconds
            cache[key] = (result, expiry_time)

            return result

        def cache_info():
            return {
                "hits": hits,
                "misses": misses
            }

        wrapper.cache_info = cache_info

        return wrapper

    return decorator


@ttl_cache(5)
def add(x, y):
    print("Calculating...")
    return x + y


print(add(x=2, y=3))
print(add(y=3, x=2))
print(add.cache_info())

Calculating...
5
5
{'hits': 1, 'misses': 1}


TASK 9

In [14]:
from datetime import datetime

malformed_count = 0


def read_errors(path):
    global malformed_count

    malformed_count = 0

    with open(path, "r") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                parts = line.strip().split("|")

                if len(parts) != 3:
                    raise ValueError

                timestamp = parts[0].strip()
                level = parts[1].strip()
                message = parts[2].strip()

                datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")

                if level == "ERROR":
                    yield (line_number, timestamp, message)

            except ValueError:
                malformed_count += 1


def summarize_errors(path):
    error_count = {}

    for line_number, timestamp, message in read_errors(path):
        date = timestamp.split(" ")[0]

        if date not in error_count:
            error_count[date] = 0

        error_count[date] += 1

    print("Errors by date:", error_count)
    print("Malformed lines:", malformed_count)


with open("app.log", "w") as file:
    file.write("2026-11-07 10:30:00 | ERROR | Connection lost\n")
    file.write("2026-11-07 11:00:00 | INFO | Server started\n")
    file.write("2026-11-07 12:00:00 | ERROR | Database failed\n")
    file.write("Invalid log line\n")


for error in read_errors("app.log"):
    print(error)

print("Malformed lines:", malformed_count)

summarize_errors("app.log")

(1, '2026-11-07 10:30:00', 'Connection lost')
(3, '2026-11-07 12:00:00', 'Database failed')
Malformed lines: 1
Errors by date: {'2026-11-07': 2}
Malformed lines: 1


TASK 10

In [15]:
import csv


def reconcile_inventory():
    inventory = {}

    with open("inventory.csv", "r") as file:
        reader = csv.DictReader(file)

        for row in reader:
            sku = row["sku"]

            inventory[sku] = {
                "name": row["name"],
                "quantity": int(row["quantity"])
            }

    rejected = []

    with open("movements.csv", "r") as file:
        reader = csv.DictReader(file)

        for row in reader:
            sku = row["sku"]
            movement_type = row["type"].upper()

            try:
                units = int(row["units"])

                if sku not in inventory:
                    raise ValueError("Unknown SKU")

                if units <= 0:
                    raise ValueError("Units must be positive")

                if movement_type == "IN":
                    inventory[sku]["quantity"] += units

                elif movement_type == "OUT":
                    if units > inventory[sku]["quantity"]:
                        raise ValueError("Insufficient stock")

                    inventory[sku]["quantity"] -= units

                else:
                    raise ValueError("Invalid movement type")

            except ValueError as error:
                rejected.append({
                    "sku": sku,
                    "type": movement_type,
                    "units": row["units"],
                    "reason": str(error)
                })

    with open("updated_inventory.csv", "w", newline="") as file:
        writer = csv.writer(file)

        writer.writerow(["sku", "name", "quantity"])

        for sku in inventory:
            writer.writerow([
                sku,
                inventory[sku]["name"],
                inventory[sku]["quantity"]
            ])

    with open("rejected_movements.csv", "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=["sku", "type", "units", "reason"]
        )

        writer.writeheader()
        writer.writerows(rejected)

    print("Updated Inventory:")

    for sku in inventory:
        print(sku, inventory[sku]["quantity"])

    print("Rejected Movements:", rejected)


with open("inventory.csv", "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(["sku", "name", "quantity"])
    writer.writerow(["P01", "Keyboard", 10])


with open("movements.csv", "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(["sku", "type", "units"])
    writer.writerow(["P01", "OUT", 4])
    writer.writerow(["P01", "OUT", 9])
    writer.writerow(["P01", "IN", 2])


reconcile_inventory()

Updated Inventory:
P01 8
Rejected Movements: [{'sku': 'P01', 'type': 'OUT', 'units': '9', 'reason': 'Insufficient stock'}]


TASK 11

In [16]:
import csv


def calculate_grade(average):
    if average >= 85:
        return "A"
    elif average >= 70:
        return "B"
    elif average >= 55:
        return "C"
    elif average >= 40:
        return "D"
    else:
        return "F"


def generate_gradebook():
    students = {}

    with open("students.csv", "r") as file:
        reader = csv.DictReader(file)

        for row in reader:
            student_id = row["student_id"]

            students[student_id] = {
                "name": row["name"],
                "scores": {}
            }

    rejected = []

    with open("scores.csv", "r") as file:
        reader = csv.DictReader(file)

        for row in reader:
            student_id = row["student_id"]
            subject = row["subject"]

            try:
                marks = float(row["marks"])

                if student_id not in students:
                    raise ValueError("Unknown student ID")

                if marks < 0 or marks > 100:
                    raise ValueError("Invalid marks")

                if subject in students[student_id]["scores"]:
                    raise ValueError("Duplicate subject")

                students[student_id]["scores"][subject] = marks

            except ValueError as error:
                rejected.append(str(error))

    report = []

    for student_id in students:
        scores = students[student_id]["scores"]

        if len(scores) > 0:
            total = sum(scores.values())
            average = total / len(scores)
        else:
            average = 0

        grade = calculate_grade(average)

        report.append({
            "student_id": student_id,
            "name": students[student_id]["name"],
            "average": round(average, 1),
            "grade": grade
        })

    report.sort(key=lambda x: (-x["average"], x["student_id"]))

    for i in range(len(report)):
        report[i]["rank"] = i + 1

    with open("report.csv", "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=["student_id", "name", "average", "grade", "rank"]
        )

        writer.writeheader()
        writer.writerows(report)

    for student in report:
        print(student)

    print("Rejected records:", rejected)


with open("students.csv", "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(["student_id", "name"])
    writer.writerow(["101", "Ali"])
    writer.writerow(["102", "Sara"])


with open("scores.csv", "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(["student_id", "subject", "marks"])
    writer.writerow(["101", "Python", 90])
    writer.writerow(["101", "Database", 80])
    writer.writerow(["101", "Networking", 85])
    writer.writerow(["102", "Python", 70])


generate_gradebook()

{'student_id': '101', 'name': 'Ali', 'average': 85.0, 'grade': 'A', 'rank': 1}
{'student_id': '102', 'name': 'Sara', 'average': 70.0, 'grade': 'B', 'rank': 2}
Rejected records: []


TASK 12

In [17]:
class ReversibleRange:
    def __init__(self, start, stop, step=1):
        if step == 0:
            raise ValueError("Step cannot be zero")

        self.start = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        current = self.start

        if self.step > 0:
            while current < self.stop:
                yield current
                current += self.step

        else:
            while current > self.stop:
                yield current
                current += self.step

    def __len__(self):
        if self.step > 0:
            return max(0, (self.stop - self.start + self.step - 1) // self.step)

        else:
            return max(0, (self.start - self.stop - self.step - 1) // (-self.step))

    def __reversed__(self):
        for i in range(len(self) - 1, -1, -1):
            yield self.start + i * self.step

    def __contains__(self, value):
        if not isinstance(value, int):
            return False

        if self.step > 0:
            if value < self.start or value >= self.stop:
                return False

        else:
            if value > self.start or value <= self.stop:
                return False

        if (value - self.start) % self.step == 0:
            return True

        return False


numbers = ReversibleRange(2, 11, 3)

print(list(numbers))
print(list(reversed(numbers)))
print(len(numbers))
print(5 in numbers)
print(6 in numbers)

[2, 5, 8]
[8, 5, 2]
3
True
False


TASK 13

In [18]:
from datetime import datetime


class BookingScheduler:
    def __init__(self):
        self.bookings = []
        self.booking_ids = []
        self.conflicts = []

    def add_booking(self, room, start, end, booking_id):
        if booking_id in self.booking_ids:
            self.conflicts.append((booking_id, "Duplicate booking ID"))
            return False

        self.booking_ids.append(booking_id)

        if end <= start:
            self.conflicts.append((booking_id, "Invalid booking time"))
            return False

        for booking in self.bookings:

            if booking["room"] == room:

                if start < booking["end"] and end > booking["start"]:
                    self.conflicts.append((booking_id, "Booking time conflict"))
                    return False

        new_booking = {
            "room": room,
            "start": start,
            "end": end,
            "booking_id": booking_id
        }

        self.bookings.append(new_booking)

        return True

    def show_bookings(self):
        sorted_bookings = sorted(
            self.bookings,
            key=lambda x: (x["room"], x["start"])
        )

        for booking in sorted_bookings:
            print(booking)

    def show_conflicts(self):
        for conflict in self.conflicts:
            print(conflict)


scheduler = BookingScheduler()

start1 = datetime(2026, 11, 7, 9, 0)
end1 = datetime(2026, 11, 7, 10, 0)

start2 = datetime(2026, 11, 7, 10, 0)
end2 = datetime(2026, 11, 7, 11, 0)

start3 = datetime(2026, 11, 7, 9, 30)
end3 = datetime(2026, 11, 7, 10, 30)


print("Booking 1:", scheduler.add_booking("A", start1, end1, 1))
print("Booking 2:", scheduler.add_booking("A", start2, end2, 2))
print("Booking 3:", scheduler.add_booking("A", start3, end3, 3))

print("\nAccepted Bookings:")
scheduler.show_bookings()

print("\nRejected Bookings:")
scheduler.show_conflicts()

Booking 1: True
Booking 2: True
Booking 3: False

Accepted Bookings:
{'room': 'A', 'start': datetime.datetime(2026, 11, 7, 9, 0), 'end': datetime.datetime(2026, 11, 7, 10, 0), 'booking_id': 1}
{'room': 'A', 'start': datetime.datetime(2026, 11, 7, 10, 0), 'end': datetime.datetime(2026, 11, 7, 11, 0), 'booking_id': 2}

Rejected Bookings:
(3, 'Booking time conflict')
